# EquiSense EDA: Market + Engineered Features

Этот ноутбук фиксирует базовые свойства данных и формулирует проверяемые гипотезы для моделирования.

**Гипотезы:**
1. Распределение доходностей имеет тяжёлые хвосты и асимметрию.
2. Технические признаки частично коллинеарны и требуют отбора/регуляризации.
3. Класс `target_up_next_day` близок к балансу, но может смещаться на рыночных режимах.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parents[0]
BACKEND = ROOT / "backend"
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

from app.features.feature_store import FeatureStore
from app.data.persistence import read_ohlcv_parquet_sync

sns.set_theme(style="whitegrid", context="talk")

In [ ]:
TICKER = "AAPL"
store = FeatureStore()

raw = read_ohlcv_parquet_sync(TICKER)
if raw is None or raw.empty:
    raise RuntimeError(f"No raw OHLCV for {TICKER}. Run refresh first.")

combined = store.build_combined(TICKER)
combined["date"] = pd.to_datetime(combined["date"])
combined = combined.sort_values("date").reset_index(drop=True)

combined["target_up_next_day"] = (combined["returns"].shift(-1) > 0).astype(int)
combined = combined.dropna(subset=["returns"]).reset_index(drop=True)

print("shape:", combined.shape)
combined.tail(3)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

sns.histplot(combined["returns"], bins=80, kde=True, ax=axes[0, 0], color="#1f77b4")
axes[0, 0].set_title("Returns distribution")

sns.histplot(combined["volatility"].dropna(), bins=60, kde=True, ax=axes[0, 1], color="#ff7f0e")
axes[0, 1].set_title("Volatility distribution")

sns.countplot(x="target_up_next_day", data=combined, ax=axes[1, 0], hue="target_up_next_day", palette="deep", legend=False)
axes[1, 0].set_title("Target balance")
axes[1, 0].set_xlabel("1 = next day up")

rolling = combined[["date", "returns"]].copy()
rolling["rolling_mean_30"] = rolling["returns"].rolling(30).mean()
rolling["rolling_vol_30"] = rolling["returns"].rolling(30).std()

sns.lineplot(data=rolling, x="date", y="rolling_mean_30", ax=axes[1, 1], label="Rolling mean (30)")
sns.lineplot(data=rolling, x="date", y="rolling_vol_30", ax=axes[1, 1], label="Rolling std (30)")
axes[1, 1].set_title("Rolling return regime features")
axes[1, 1].set_ylabel("value")

plt.tight_layout()

In [ ]:
key_cols = [
    "returns", "volatility", "rsi", "macd", "macd_signal", "sma_20", "sma_50",
    "bb_width", "momentum", "target_up_next_day"
]
corr = combined[key_cols].corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title(f"Feature correlation heatmap ({TICKER})")
plt.tight_layout()

In [ ]:
na_ratio = (combined.isna().mean() * 100).sort_values(ascending=False)
na_ratio = na_ratio[na_ratio > 0]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
if len(na_ratio) > 0:
    sns.barplot(x=na_ratio.index, y=na_ratio.values, palette="rocket", ax=axes[0])
    axes[0].tick_params(axis="x", rotation=70)
    axes[0].set_ylabel("Missing %")
    axes[0].set_title("Missingness by feature")
else:
    axes[0].text(0.5, 0.5, "No missing values", ha="center", va="center", fontsize=14)
    axes[0].axis("off")

abs_ret = combined["returns"].abs().dropna()
q = np.linspace(0, 1, len(abs_ret), endpoint=False)
emp = np.quantile(abs_ret, q)
gauss = np.quantile(np.abs(np.random.normal(loc=0, scale=abs_ret.std(), size=200000)), q)

axes[1].scatter(gauss, emp, s=8, alpha=0.5)
lim = max(float(np.nanmax(gauss)), float(np.nanmax(emp)))
axes[1].plot([0, lim], [0, lim], "r--", lw=1)
axes[1].set_title("Tail heaviness check (|returns| vs Gaussian)")
axes[1].set_xlabel("Gaussian |returns| quantiles")
axes[1].set_ylabel("Empirical |returns| quantiles")

plt.tight_layout()

## Выводы по EDA

- Доходности и волатильность показывают выраженную нестационарность по времени (rolling-фичи меняют режимы).
- Корреляционная матрица подтверждает мультиколлинеарность части индикаторов (`sma`, `macd`-семейство, `bb_width`).
- Целевая переменная не полностью вырожденная, поэтому задача классификации корректна.
- Проверка хвостов показывает отклонение от гауссовости, что объясняет нестабильность простых линейных моделей.

**Рекомендация:** в моделировании использовать time-series CV, регуляризацию и сравнение с нелинейными моделями.